In [1]:
from common import *
u, s = joblib.load(RES / "unsup.pkl"), joblib.load(RES / "sup.pkl")
PRED, SCORE = {**u["PRED"], **s["PRED"]}, {**u["SCORE"], **s["SCORE"]}
R = pd.DataFrame([metrics(n, yte, PRED[n], SCORE[n]) for n in PRED]).set_index("model").round(3).sort_values("macroF1", ascending=False)
print(R); R.to_csv(RES / "results.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\Prisha\\Downloads\\ml_turbofan\\data preprocessing\\processed_output\\run_20261029_201358_328692\\results\\unsup.pkl'

In [ ]:
show = ["CANARI", "Mahalanobis (d=1)", "Isolation Forest", "Random Forest"]
fig, ax = plt.subplots(1, 4, figsize=(18, 4))
for a, n in zip(ax, show):
    ConfusionMatrixDisplay.from_predictions(yte, PRED[n], labels=["normal", "near_anomaly", "anomaly"],
                                            normalize="true", ax=a, colorbar=False, xticks_rotation=30)
    a.set_title(n)
plt.tight_layout(); plt.show()

In [ ]:
def lead(pred, k=3):    # RUL at the first point where k consecutive cycles are flagged (bigger = earlier)
    df = test[["unit", "cycle", "rul"]].assign(f=(pred != "normal")).sort_values(["unit", "cycle"])
    out = []
    for _, g in df.groupby("unit"):
        run = g.f.rolling(k).sum().values == k
        out.append(g.rul.values[np.argmax(run)] if run.any() else np.nan)
    s = pd.Series(out)
    return dict(median_lead_cycles=s.median(), missed_engines=s.isna().mean())

print(pd.DataFrame({n: lead(PRED[n]) for n in PRED}).T.round(2))

In [ ]:
b = pd.cut(test["rul"].clip(upper=150), bins=range(0, 160, 10))
for n in ["CANARI", "Mahalanobis (d=1)", "Isolation Forest", "Autoencoder"]:
    s = pd.Series(SCORE[n]); z = ((s - s.mean()) / s.std()).groupby(b.values).mean()
    plt.plot([i.mid for i in z.index], z.values, label=n)
plt.gca().invert_xaxis(); plt.xlabel("RUL (cycles)"); plt.ylabel("mean z-scored score"); plt.legend(); plt.show()

In [ ]:
print(pd.DataFrame({n: {k: f1_score(yte[test.subset == k], PRED[n][test.subset == k], average="macro")
      for k in sorted(test.subset.unique())} for n in ["CANARI", "Isolation Forest", "Random Forest"]}).round(3))

In [ ]:
def relabel(rul, A, B): return np.where(rul > A, "normal", np.where(rul > B, "near_anomaly", "anomaly"))
rows = []
for A in (40, 60, 80):
    for B in (10, 15, 25):
        y2 = relabel(test["rul"].values, A, B)
        rows.append(dict(A=A, B=B, **{n: round(f1_score(y2, PRED[n], average="macro"), 3)
                                      for n in ["CANARI", "Isolation Forest", "Random Forest"]}))
print(pd.DataFrame(rows))